# Logistic Regression

Cox (1958): https://www.jstor.org/stable/2983890



Elastic Net (2005): https://rss.onlinelibrary.wiley.com/doi/10.1111/j.1467-9868.2005.00503.x

# Logistic Regression

In [1]:
# Arabic Sentiment Analysis — Logistic Regression (Cox, 1958)


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

# Fit TF-IDF ONLY on training data — strict no-leakage policy
tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))
print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    """Convert emoji features to sparse matrix for hstack."""
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# LogisticRegression with solver='saga' supports sparse input natively
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])
print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Logistic Regression (L2, Cox 1958)
# solver='saga'   : supports L1/L2/ElasticNet + sparse inputs
# C=1.0           : inverse regularisation strength (higher = less regularised)
# class_weight    : compensates for label imbalance automatically
# max_iter=1000   : enough for convergence on TF-IDF data
print("=" * 60)
print("  📈 Training: Logistic Regression (L2 / Cox, 1958)")
print("=" * 60)

clf = LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        solver='lbfgs',
        C=1.0,
        class_weight='balanced',
        multi_class='auto'
)
clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Logistic Regression (L2)  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  📈 Training: Logistic Regression (L2 / Cox, 1958)

────────────────────────────────────────────────────────────
  Logistic Regression (L2)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8297
  Precision : 0.8297
  Recall    : 0.8297
  F1-Score  : 0.8296

  Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.83      0.83       543
           1       0.83      0.83      0.83       543

    accuracy                           0.83      1086
   macro avg       0.83      0.83      0.83      1086
weighted avg       0.83      0.83      0.83      1086

  Confusion Matrix:
[[449  94]
 [ 91 452]]


─────────────────────────────────

# Logistic Regression with Elastic Net

In [2]:
# Arabic Sentiment Analysis — Elastic Net Logistic Regression (Zou & Hastie, 2005)


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# Elastic Net mixing ratio: 0 = pure L2 (Ridge), 1 = pure L1 (Lasso)
# 0.15 keeps most L2 stability while adding moderate L1 sparsity
L1_RATIO = 0.15
C        = 1.0    # inverse regularisation strength

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))
print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# saga solver + elasticnet penalty support sparse matrices directly
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])
print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Elastic Net Logistic Regression
# penalty='elasticnet' requires solver='saga' — the only solver that
# implements the full Elastic Net proximal gradient update.
print("=" * 60)
print(f"  📈 Training: Elastic Net Logistic Regression (l1_ratio={L1_RATIO})")
print("=" * 60)

clf = LogisticRegression(
    penalty      = "elasticnet",
    C            = C,
    l1_ratio     = L1_RATIO,
    solver       = "saga",          # required for elasticnet
    max_iter     = 1000,
    class_weight = "balanced",
    n_jobs       = -1,
    random_state = RANDOM_STATE,
)
clf.fit(X_train, y_train)

# Report sparsity of learned weights
n_weights = clf.coef_.size
n_zero    = (clf.coef_ == 0).sum()
print(f"\n  Weight sparsity : {n_zero}/{n_weights} ({100*n_zero/n_weights:.1f}%) zeroed by L1\n")

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Elastic Net LR  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  📈 Training: Elastic Net Logistic Regression (l1_ratio=0.15)

  Weight sparsity : 2820/7728 (36.5%) zeroed by L1


────────────────────────────────────────────────────────────
  Elastic Net LR  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8241
  Precision : 0.8241
  Recall    : 0.8241
  F1-Score  : 0.8241

  Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.83      0.82       543
           1       0.83      0.82      0.82       543

    accuracy                           0.82      1086
   macro avg       0.82      0.82      0.82      1086
weighted avg       0.82      0.82      0.82      1086

  Confusion Matrix:
[[44